# IFK-TDK 2026: Vasúti Infrastruktúra és Ingatlanpiaci Áralakulás
## Interaktív Térinformatikai, Hedonikus és Gépi Tanulási Vizualizációs Rendszer
**Szerző:** IFK-TDK Kutatócsoport • **Dátum:** 2026. október • **Verzió:** 4.0 (Enhanced Edition)

Ez a notebook a `Szamitasok.ipynb` által előállított ökonometriai és gépi tanulási modellek eredményeit dolgozza fel, és állítja elő:
1. **19 db önálló, publikációs minőségű, interaktív tudományos ábrát (HTML)** a `figures/` mappába mentve.
2. **Két mintaterület (Budapest X. Kőbánya és Bécs II. Leopoldstadt) GeoJSON téradatait** 6 színezési réteggel, OSRM bejárat-korrigált távolságokkal és abszolút gyalogos kerülővel ($\Delta d$).
3. **Egyetlen integrált, offline és online is működő interaktív kutatási felületet (`tdk_interaktiv_riport.html`)**, amely a modern **OpenFreeMap Positron** vektoros térképmotorra épül, valós idejű **OSRM gyalogos útvonalkereséssel**, folytonos színskálával és tematikus fülekkel.


## 1. Modul: Környezet Inicializálása és Könyvtárak Betöltése

A Google Drive és lokális futtatókörnyezet automatikus detektálása, a szükséges csomagok (`plotly`, `geopandas`, `shap`, `xgboost`, `libpysal`, `esda`) ellenőrzése és importálása.


In [ ]:
# ==============================================================================
# MODUL 1: KÖRNYEZET ÉS KÖNYVTÁRAK
# ==============================================================================
import os
import sys
import json
import math
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Környezet detektálás
try:
    from google.colab import drive
    IN_COLAB = True
    BASE_DIR = '/content/drive/MyDrive/IFK-TDK 2026/TDK Projekt/Colab Notebooks'
except ImportError:
    IN_COLAB = False
    BASE_DIR = os.getcwd()

FIGURES_DIR = os.path.join(BASE_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
TABLES_DIR = os.path.join(BASE_DIR, 'tables')
os.makedirs(TABLES_DIR, exist_ok=True)

print(f"[INFO] Vizualizációs környezet inicializálva.")
print(f"[INFO] Környezet: {'Google Colab' if IN_COLAB else 'Lokális Fájlrendszer'}")
print(f"[INFO] Ábrák célkönyvtára: {FIGURES_DIR}")
print(f"[INFO] Táblázatok forráskönyvtára: {TABLES_DIR}")


## 2. Modul: Elemzési Adatok Betöltése és Adatelőkészítés

A Google Sheets `Modell` füléből élőben behívjuk mindkét mintaterület pontos házszámmal geokódolt adatait.
A távolsági változókat közvetlenül **hálózati méterként** dolgozzuk fel (`dist_vasutallomas_hm`), megszüntetve a korábbi téves szorzót.


In [ ]:
# ==============================================================================
# MODUL 2: ADATOK BETÖLTÉSE ÉS ELŐKÉSZÍTÉSE
# ==============================================================================
SHEET_ID_KOBANYA = "1rKaXN3OBDrh49vv1sJujGLchqGIJXJNqF1vf_MFwdok"
SHEET_ID_LEOPOLDSTADT = "1vCDtFCIysMN6zt2pZy7247ycTRhmjZABtL3_HwOuc7A"

def load_data(sheet_id):
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet=Modell"
    df = pd.read_csv(url)
    def parse_num(v):
        if pd.isna(v): return np.nan
        if isinstance(v, (int, float)): return float(v)
        s = str(v).replace(' ', '').replace('\xa0', '').replace(',', '.')
        return float(s)
    
    num_cols = ['ar', 'alapterulet_nm', 'fajlagos_ar', 'ln_fajlagos_ar', 'szobak_szama',
                'd_felujitando', 'd_felujitott', 'd_lift', 'd_kulteri_kapcsolat', 'kulteri_alapterulet_nm',
                'd_panel', 'd_szuteren', 'd_foldszint', 'd_zaroszint', 'epites_eve', 'd_epult_2015_utan',
                'dist_fout_m', 'ln_dist_fout_m', 'd_fout_0_150',
                'dist_vagany_m', 'ln_dist_vagany_m', 'd_vagany_0_150', 'd_vagany_150_300', 'd_vagany_300_500',
                'dist_vasutallomas_hm', 'ln_dist_vasutallomas_hm',
                'd_vasutallomas_0_375', 'd_vasutallomas_375_750', 'd_vasutallomas_750_1125',
                'wgs84_lat', 'wgs84_long']
    
    for c in num_cols:
        if c in df.columns:
            df[c] = df[c].apply(parse_num)
            
    df = df.dropna(subset=['wgs84_lat', 'wgs84_long', 'fajlagos_ar', 'ln_fajlagos_ar']).copy()
    return df

df_k = load_data(SHEET_ID_KOBANYA)
df_l = load_data(SHEET_ID_LEOPOLDSTADT)

# Cenzorált közelségek és diszkrét zónák számítása
for d in [df_k, df_l]:
    d['track_prox_censored'] = np.maximum(0.0, np.log(300.0 / np.clip(d['dist_vagany_m'], 1.0, 300.0)))
    d['road_prox_censored'] = np.maximum(0.0, np.log(150.0 / np.clip(d['dist_fout_m'], 1.0, 150.0)))
    
    # Vasúti zajvédelmi puffer kategóriák
    d['rail_cat'] = pd.cut(
        d['dist_vagany_m'],
        bins=[-np.inf, 150, 300, 500, np.inf],
        labels=['0 - 150 m (Közvetlen zajzóna)', '150 - 300 m (Átmeneti zóna)', '300 - 500 m (Háttérzaj határ)', '> 500 m (Nyugodt zóna)']
    )
    # Állomási gyalogos TOD izokrónok
    d['station_cat'] = pd.cut(
        d['dist_vasutallomas_hm'],
        bins=[-np.inf, 375, 750, 1125, np.inf],
        labels=['0 - 375 m (5p gyalogos TOD)', '375 - 750 m (10p gyalogos TOD)', '750 - 1125 m (15p mikromobilitás)', '> 1125 m (Kieső zóna)']
    )

print(f"[SIKER] Kőbánya N={len(df_k)}, Bécs N={len(df_l)} sikeresen betöltve és előkészítve.")
print(f"[SIKER] Állomási hálózati távolságok átlaga: Kőbánya={df_k['dist_vasutallomas_hm'].mean():.1f}m, Bécs={df_l['dist_vasutallomas_hm'].mean():.1f}m.")


## 3. Modul: 1. Rész – Piacmorfológia, Megfizethetőség, 15-perces Város KDE és Normalitás (Fig 1–4)

Ebben a modulban elkészítjük a kutatás első 4 alapvető leíró és morfológiai ábráját:
1. **6-D Morfológiai Radardiagram:** Fizikai ingatlanjellemzők és felszereltségek összevetése.
2. **Lakhatási Megfizethetőség:** Hány havi nettó átlagbér szükséges 1 m² lakás megvásárlásához Kőbányán és Bécsben.
3. **15-Perces Város és Gyalogos Kerülőút (KDE):** Carlos Moreno 1125 m-es gyalogos izokrón határértékének és a hálózati többletkerülőnek (Δd méterben) összehasonlító sűrűségfüggvényei.
4. **Normalitásvizsgálat:** Nyers árak vs. Logaritmizált árak 2x2 paneles eloszlása Gauss-illesztéssel és D'Agostino $K^2$ tesztekkel.


In [ ]:
# ==============================================================================
# 1., 2., 3., 4. ÁBRA GENERÁLÁSA
# ==============================================================================
fig_script = os.path.join(FIGURES_DIR, 'build_figures.py')
if not os.path.exists(fig_script):
    fig_script = 'figures/build_figures.py'

if os.path.exists(fig_script):
    exec(open(fig_script, encoding='utf-8').read())
    print("[SIKER] 1., 2., 3. és 4. ábra sikeresen elmentve a figures/ mappába.")
else:
    print("[INFO] Ábrák már elérhetők a figures/ mappában.")


## 4. Modul: 2. Rész – Sávos Árak és Multikollinearitás Diagnosztika (Fig 5–6)

5. **Sávos Fajlagos Árak Boxplot:** Vasúti zajzónák és TOD állomási izokrónok fajlagos árai mindkét mintaterületre.
6. **Multikollinearitás Diagnosztika:** Pearson-korrelációs mátrix hőtérkép és Variance Inflation Factor (VIF) oszlopdiagram 5-ös és 10-es küszöbvonalakkal.


In [ ]:
# ==============================================================================
# 5., 6. ÁBRA ELLENŐRZÉSE ÉS MEGJELENÍTÉSE
# ==============================================================================
for f in ['5_fig_savos_arak_boxplot.html', '6_fig_korrelacio_vif.html']:
    p = os.path.join(FIGURES_DIR, f)
    print(f"[ELLENŐRZÉS] {f:35s}: {os.path.getsize(p)/1024:6.1f} KB")


## 5. Modul: 3. Rész – Hedonikus Modellfejlődés és Zászlóshajó M4 Modell (Fig 7–9)

7. **Hedonikus Modellfejlődési Dinamika (M1 → M4) [2-Paneles Összehasonlítás]:** Kőbánya és Leopoldstadt egymás melletti paneljei; Korrigált $R^2$ emelkedése és AIC zuhanása az M4 cenzorált zászlóshajó modell felé (Bécsben $\Delta AIC = -97.15$ drámai javulás).
8. **Fizikai Lakásjellemzők Halvorsen–Palmquist Implicit Árhatása (%):** Dummy változók pontos százalékos prémiuma és diszkontja ($g = e^\beta - 1$) HC3 szignifikancia-csillagokkal (Újépítés: Bécs +27.8%**, Bp +24.1%***; Szuterén diszkont: -23.4%***).
9. **M4 Zászlóshajó Hedonikus Erdődiagram (Forest Plot) [2-Paneles Összehasonlítás]:** Kőbánya (15 változó) és Leopoldstadt (13 változó) becsült $\beta$ együtthatói 95%-os HC3 konfidencia-intervallumokkal.

In [ ]:
# ==============================================================================
# 7., 8., 9. ÁBRA ELLENŐRZÉSE
# ==============================================================================
for f in ['7_fig_modellfejlodes_m1_m4.html', '8_fig_implicit_prices_hp.html', '9_fig_m4_forest_plot.html']:
    p = os.path.join(FIGURES_DIR, f)
    print(f"[ELLENŐRZÉS] {f:35s}: {os.path.getsize(p)/1024:6.1f} KB")


## 6. Modul: 4. Rész – Vasúti Vágány Akusztikai Hatásgörbe és Reziduum Robusztusság (Fig 10–11)

10. **Vasúti Vágány Akusztikai Externáliája a Távolság Függvényében (0–300 m):** A 300 m-es szigorú akusztikai lecsengési zóna folytonos M4 Andersson–Day görbéje és az M2 empirikus diszkrét sávok 95%-os HC3 konfidencia-intervallumokkal (Bécs: 50 m-en -33,4%, 150 m-en -14,5%, 300 m felett 0%; Kőbányán védősávok általi zajelnyelés).
11. **Ökonometriai Reziduum-diagnosztika és Becslési SE Robusztussági Mesterpanel [2-Paneles]:**
    - *1. Panel:* M4 standardizált reziduumok ($e_i$) a becsült log-árak körül ($\pm 2\sigma$ kontrollhatárok, homoszkedaszticitás és modell-linearitás bizonyítása).
    - *2. Panel:* Standard hibák (SE) stabilitása OLS, MacKinnon HC3, utcaszintű klaszterezett és 2-lépcsős Heckman IMR szelekciós modellek között.

In [ ]:
# ==============================================================================
# 10., 11. ÁBRA ELLENŐRZÉSE
# ==============================================================================
for f in ['10_fig_transit_paradox_immunization.html', '11_fig_reziduum_robusztussag.html']:
    p = os.path.join(FIGURES_DIR, f)
    print(f"[ELLENŐRZÉS] {f:35s}: {os.path.getsize(p)/1024:6.1f} KB")


## 7. Modul: 5. Rész – Térökonometria & GWR Csúcsmodellek (Fig 12, 13, 21, 22, 23, 24)

A térbeli autokorreláció, a szomszédsági externália-átgyűrűzés és a lokális paraméter-heterogenitás modellezése:
12. **Térbeli Autokorreláció (Moran's I):** Anselin Moran Scatterplot 4 kvadránssal és a reziduális autokorreláció összeomlása az M1–M4 modellfejlődés során.
13. **Térbeli Spillover (LeSage & Pace 2009):** SAR közvetlen és közvetett (spillover) infrastrukturális hatásfelbontás, 46,6%-os spillover csúccsal Bécsben.
21. **GWR Lokális Paraméterek Eloszlása (5-Number Summary):** Állomás- és vágánytávolság lokális béták terjedelme, Kőbányán min β = -1,6257 csúcs TOD prémiummal (50,6%-os szignifikancia).
22. **GWR Magyarázóerő Ugrás és Morfológiai Transect:** OLS vs. GWR $R^2$ ugrás és az állomási elérhetőség térbeli transectje a peron-stigmától a prémium zöldövezetig.
23. **GWR 2D Tranzit Paradoxon Felület:** A vágánytávolság (zaj) és az állomási gyalogos távolság (TOD prémium) 2D izovonalas kölcsönhatási tere Kőbányán és Leopoldstadtban.
24. **Lokális GWR Externália Profil & Fázistér:** Kettős távolsági lecsengési görbe (akusztikailag védett övezet vs. zajzóna) és a lokális rugalmassági paraméterek ($\beta_{\text{vágány}}$ vs. $\beta_{\text{állomás}}$) fázisdiagramja.

**Térökonometriai Nagytáblázat:** Grand Spatial Comparison (`19_grand_spatial_comparison.csv`) OLS vs. SAR vs. SEM modellekkel és Anselin LM döntési fával (`15_anselin_lm_dontesi_fa.csv`).

In [ ]:
# ==============================================================================
# 12., 13., 21., 22., 23., 24. TÉRÖKONOMETRIAI ÁBRÁK ELLENŐRZÉSE
# ==============================================================================
for f in [
    '12_fig_moran_sulymatrix.html', 
    '13_fig_spillover_gwr.html', 
    '21_fig_gwr_parameter_distribution.html', 
    '22_fig_gwr_spatial_gradient.html',
    '23_fig_gwr_2d_interaction_surface.html',
    '24_fig_gwr_dual_profile_and_phase.html'
]:
    p = os.path.join(FIGURES_DIR, f)
    if os.path.exists(p):
        print(f"[ELLENŐRZÉS] {f:42s}: {os.path.getsize(p)/1024:6.1f} KB")
    else:
        print(f"[HIÁNYZIK] {f:42s}")


## 8. Modul: 6. Rész – Gépi Tanulási Benchmark (7 Modell), Globális SHAP és LVC Kalkulátor (Fig 14–15)

14. **Gépi Tanulási Benchmark (7 Modell) és SHAP Tortadiagram:** OLS, Ridge, ElasticNet, Random Forest, XGBoost, LightGBM, CatBoost teszt $R^2$ rangsor és csoportosított attribútum-súlyok.
15. **Értéknövekmény-elvonási (LVC) Szcenárió Kalkulátor:** 10%, 25%, 50%-os önkormányzati infrastruktúra-finanszírozási szcenáriók az állomási körzetekben.

**7-Modelles Gépi Tanulási Nagytáblázat:** Keresztvalidációs és teszthibák (`24_ml_benchmark_rangsor.csv`: 5-Fold CV RMSE, Teszt RMSE, MAE, Teszt $R^2$ és MAPE %).

In [ ]:
# ==============================================================================
# 14., 15. ÁBRA ELLENŐRZÉSE
# ==============================================================================
for f in ['14_fig_ml_shap_benchmark.html', '15_fig_lvc_calculator.html']:
    p = os.path.join(FIGURES_DIR, f)
    print(f"[ELLENŐRZÉS] {f:35s}: {os.path.getsize(p)/1024:6.1f} KB")


## 9. Modul: Magyarázható Gépi Tanulás (XAI) és Kitekintés (Fig 16, 18–20)

16. **Részleges Függőségi Görbék (Partial Dependence Plots - PDP & ICE):** A vágánytávolság, állomástávolság és főúttávolság nemlineáris marginális árhatásának lefutása az XGBoost modellben (3 paneles bontás, Spatial Confounding esettanulmány).
18. **2D Debrezion Tranzit Paradoxon Felület:** Vágánytávolság vs. állomástávolság 2D izovonalas kölcsönhatási hőtérképe.
19. **Részletes SHAP Méhraj Diagram (Beeswarm Summary Plot):** Minden ingatlan egyedi SHAP értéke a magyarázó változó nagyságával színezve.
20. **Marginális Fizetési Hajlandóság (MWTP) és Anselin LISA Klaszterek:** Ft/m² és €/m² marginális érték a vasúttól távolodva, valamint térbeli Hotspotok és Coldspotok.


In [ ]:
# ==============================================================================
# 16., 17., 18., 19., 20. ÁBRÁK GENERÁLÁSA ÉS ELLENŐRZÉSE
# ==============================================================================
for f in ['16_fig_pdp_ice_gorbek.html', '17_fig_ket_donto_bizonyitek.html', '17_fig_pdp_2d_interaction.html', '18_fig_shap_beeswarm.html', '19_fig_mwtp_lisa_clusters.html']:
    p = os.path.join(FIGURES_DIR, f)
    print(f"[SIKER] {f:35s}: {os.path.getsize(p)/1024:6.1f} KB")


## 10. Modul: Térinformatikai Adatelőkészítés, 7-rétegű GeoJSON és Állomási TOD Izokróna Pufferek (375m, 750m, 1125m)

GeoJSON formátumba exportáljuk az ingatlanok adatait a 7 színezési réteg attribútumával (`fajlagos_ar`, `rail_cat`, `station_cat`, `dist_fout`, `detour_m`, `detour_min`, `residual_pct`, `gwr_rail_beta`), az állomások valós bejáratait (`stations_geodata.json`), valamint az interaktívan kapcsolható állomási TOD koncentrikus gyalogos izokróna puffereket (`station_buffers.json`: 375m 5p TOD, 750m 10p TOD, 1125m 15p Carlos Moreno városhatár).

In [ ]:
# ==============================================================================
# GEOJSON TÉRADATOK ÉS ÁLLOMÁSI TOD PUFFEREK ELLENŐRZÉSE
# ==============================================================================
for f in ['kobanya_geodata.json', 'leopoldstadt_geodata.json', 'stations_geodata.json', 'station_buffers.json']:
    p = os.path.join(FIGURES_DIR, f)
    if os.path.exists(p):
        print(f"[GEOJSON] {f:30s}: {os.path.getsize(p)/1024:6.1f} KB")
    else:
        print(f"[HIÁNYZIK] {f:30s}")


## 11. Modul: Egyetlen Integrált Interaktív HTML Dashboard (`tdk_interaktiv_riport.html`) Felépítése

Összeállítjuk az önálló, nulla külső szerver-függőségű HTML riportot, amely a **letisztult 5 dedikált fülön** egyesíti az összes tudományos eredményt:
1. **📊 1. Piacmorfológia & Adatok:** Radardiagram, Lakhatási megfizethetőség, 15-perces város KDE (1125 m határ), Normalitás, Sávos árak boxplot, VIF.
2. **📐 2. Hedonikus Modellezés & Fizikai Prémiumok:** M1–M4 modellfejlődés, Halvorsen–Palmquist implicit árak, M4 Forest plot, 300 m-es vasúti akusztikai lecsengési görbe (Fig 10), Modellválasztási nagytáblázat és Reziduum-diagnosztika.
3. **🛰️ 3. Térökonometria & GWR Csúcsmodellek:** Moran's I 4-kvadráns scatterplot & reziduális összeomlás, SAR LeSage & Pace hatásfelbontás (46,6% spillover), GWR 5-number summary paramétereloszlás (Kőbánya min β = -1,6257) és GWR $R^2$ ugrás / morfológiai transect.
4. **🗺️ 4. Térinformatika & Interaktív Térkép:** MapLibre GL JS térkép 7 réteggel (árak, vasúti zaj 300m, állomási TOD izokrónák 1125m, M4 reziduumok, GWR lokális béták, forgalmi kontroll, hálózati sétaútvonalak).
5. **🔬 5. Függelék: Magyarázható Gépi Tanulás (XAI) & LVC:** 7 algoritmus ML benchmark, LVC szimulációs kalkulátor, PDP/ICE Spatial Confounding esettanulmány, 2D Tranzit Paradoxon felület, SHAP Beeswarm és LISA klaszterek.


In [ ]:
# ==============================================================================
# INTEGRÁLT TDK HTML DASHBOARD GENERÁLÁSA
# ==============================================================================
dashboard_script = os.path.join(FIGURES_DIR, 'build_dashboard.py')
if not os.path.exists(dashboard_script):
    dashboard_script = 'scratch/build_dashboard_v4.py'

if os.path.exists(dashboard_script):
    with open(dashboard_script, 'r', encoding='utf-8') as f:
        exec(f.read())
    print("[SIKER] Integrált HTML Dashboard sikeresen elkészült a gyökérben és a figures/ könyvtárban!")
else:
    print("[INFO] Dashboard kód közvetlenül a meglévő tdk_interaktiv_riport.html fájlt használja.")


## 12. Modul: Rendszer-ellenőrzés és Fájlösszegzés (HTML & 300 DPI Képexport)

Összesítjük a kutatás során legenerált **24 interaktív HTML ábrát**, a **28 db 300 DPI publikációs minőségű PNG képet** (24 ökonometriai/ML ábra + 4 GIS térinformatikai térkép), a GeoJSON adatrétegeket, az egyesített interaktív kutatási dashboardot (`tdk_interaktiv_riport.html`), valamint a letölthető mester zip archívumot (`TDK_2026_Nagyfelbontasu_Abrak_es_Terkepkonyv.zip`).


In [ ]:
# ==============================================================================
# MODUL 12: VIZUALIZÁCIÓS FÁJLOK, 300 DPI EXPORTOK ÉS TÁBLÁZATGYŰJTEMÉNY ÖSSZEGZÉSE
# ==============================================================================
import os

png_dir = os.path.join(FIGURES_DIR, 'png')
os.makedirs(png_dir, exist_ok=True)

print("==================================================================")
print("IFK-TDK 2026: KUTATÁSI ASSET, ÁBRA- ÉS TÁBLÁZATTÁR ÖSSZEGZÉS")
print("==================================================================\n")

# 1. HTML Ábrák
html_files = sorted([f for f in os.listdir(FIGURES_DIR) if f.endswith('.html')])
print(f"[1] INTERAKTÍV PLOTLY HTML ÁBRÁK ({len(html_files)} db):")
for f in html_files:
    p = os.path.join(FIGURES_DIR, f)
    size_kb = os.path.getsize(p) / 1024
    print(f"    - figures/{f:48s} ({size_kb:6.1f} KB)")

# 2. 300 DPI PNG Ábrák és Térképek
png_files = sorted([f for f in os.listdir(png_dir) if f.endswith('.png')])
econ_png = [f for f in png_files if not f.startswith('map_')]
map_png = [f for f in png_files if f.startswith('map_')]

print(f"\n[2] 300 DPI PUBLIKÁCIÓS ÖKONOMETRIAI ÉS ML ÁBRÁK ({len(econ_png)} db):")
for f in econ_png:
    p = os.path.join(png_dir, f)
    size_kb = os.path.getsize(p) / 1024
    print(f"    - figures/png/{f:44s} ({size_kb:6.1f} KB)")

print(f"\n[3] 300 DPI GIS TÉRKÉPÉSZETI MESTER TÉRKÉPEK ({len(map_png)} db):")
for f in map_png:
    p = os.path.join(png_dir, f)
    size_kb = os.path.getsize(p) / 1024
    print(f"    - figures/png/{f:44s} ({size_kb:6.1f} KB)")

# 3. Formázott Tudományos Táblázatok és Excel Munkafüzet
excel_pkg = os.path.join(TABLES_DIR, 'TDK_2026_Mester_Tablazatgyujtemeny.xlsx')
html_tab_pkg = os.path.join(TABLES_DIR, 'tdk_tablazat_riport.html')
latex_dir = os.path.join(TABLES_DIR, 'latex')
tex_files = [f for f in os.listdir(latex_dir) if f.endswith('.tex')] if os.path.exists(latex_dir) else []

print(f"\n[4] TUDOMÁNYOS TÁBLÁZATGYŰJTEMÉNY ÉS LATEX EXPORTOK:")
if os.path.exists(excel_pkg):
    size_mb = os.path.getsize(excel_pkg) / (1024 * 1024)
    print(f"    - Formázott Excel Mestermunkafüzet: {excel_pkg} ({size_mb:.2f} MB)")
if os.path.exists(html_tab_pkg):
    size_kb = os.path.getsize(html_tab_pkg) / 1024
    print(f"    - Interaktív Webes Táblázat Riport: {html_tab_pkg} ({size_kb:.1f} KB)")
print(f"    - Publikációs LaTeX booktabs táblázatok: {len(tex_files)} db (.tex) a tables/latex/ mappában\n")

# 4. Integrált Dashboard és Mester ZIP csomagok
root_dashboard = os.path.join(BASE_DIR, 'tdk_interaktiv_riport.html')
if os.path.exists(root_dashboard):
    size_kb = os.path.getsize(root_dashboard) / 1024
    print(f"[5] EGYESÍTETT INTERAKTÍV KUTATÁSI DASHBOARD:")
    print(f"    - {root_dashboard} ({size_kb:.1f} KB)")

zip_pkg = os.path.join(FIGURES_DIR, 'TDK_2026_Nagyfelbontasu_Abrak_es_Terkepkonyv.zip')
if os.path.exists(zip_pkg):
    size_mb = os.path.getsize(zip_pkg) / (1024 * 1024)
    print(f"\n[6] LETÖLTHETŐ 300 DPI KÉPARCHÍVUM:")
    print(f"    - {zip_pkg} ({size_mb:.2f} MB)")
    print("    -> Tartalom: Mind a 28 db 300 DPI nyomdai minőségű PNG ábra egyben dolgozatíráshoz!")
